# 07 - Training on a harder domain: SMAClite

So far the tasks were small: 2-6 identical agents moving dots around (MPE). **SMAClite** (notebook 06) is a step up, and the kind of
benchmark cooperative MARL algorithms are measured on:
* 5 to 10 agents of **different unit types** - ranged Stalkers, melee Zealots, Marines, Marauders, and a Medivac that can only heal,
* **partial observation** (each unit sees only what is near it), **discrete actions** with availability masks,
* units that **die** mid-battle, and battles that are won or lost,
* **one team reward** per step (damage dealt, kills, win) shared by every agent.

We train three algorithms on two maps - `2s3z` (3 random seeds each) and `MMM` (1 seed) - and watch what they learn:
* **MAPPO** - policy gradient with a centralised critic (notebook 04, now with discrete actions),
* **VDN** and **QMIX** - value-based methods that build the team's value from per-agent values (notebook 05, in their original discrete form).

All three use the same small networks (2 hidden layers of 64) and see, per agent: its observation, its last action, its id and the time left.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import load_run
from marl.smac_analysis import *
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90
COL = {"mappo": "tab:blue", "vdn": "tab:orange", "qmix": "tab:green"}

## 1. MAPPO

* **Actor** (shared by all agents): picks a discrete action from the agent's own input; actions that are not available are masked out.
* **Critic** V(state, agent id): how good the situation is, judged from the global state (used only during training).
* Each agent's action is reinforced in proportion to its **advantage**: (team rewards that followed) - V(state, i).
* Training data comes from 16 battles running in parallel worker processes.

In [ ]:
mappo = seed_runs("2s3z", "mappo")
describe_smac_networks(mappo[1])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
plot_seeds(mappo, "train_win_rate", ax[0], "MAPPO", COL["mappo"]); ax[0].set_title("2s3z: battles won while training")
plot_seeds(mappo, "return", ax[1], "MAPPO", COL["mappo"]); ax[1].set_title("2s3z: return of the greedy policy (max 20)")
for a in ax: a.legend()
plt.tight_layout()

MAPPO wins `2s3z` - fast in two of the three seeds. One seed got stuck (killing most enemies, never winning), then learned, then
**collapsed** at 0.8M steps: its policy had become almost deterministic and never recovered.

In [ ]:
show_video(smac_evolution(mappo[1], k=4), fps=6, width=720)

## 2. VDN: split the team's value into a sum

Same idea as the VDN critic in 03b, now with discrete actions. Each agent has a **utility** Q_i(input_i, action_i) for every one of its
actions (one shared network, with the agent id as input), and the team's value is the sum:

$$Q_{tot} = Q_1 + Q_2 + \dots + Q_N$$

Only Q_tot is fitted to the team reward; the error flows back into each Q_i. Each agent simply picks **argmax Q_i** - no separate actor network -
and because Q_tot is a sum, that is also the best joint action for the team.

In [ ]:
vdn = seed_runs("2s3z", "vdn")
describe_smac_networks(vdn[0])

## 3. QMIX: a learned, monotonic mix

As in 03b: a **mixing network** combines the Q_i, with weights produced from the global state by **hypernetworks**
(e.g. "right now the agent that is low on health matters more"). The weights are kept **positive**, so Q_tot rises whenever any Q_i rises:
each agent picking its own argmax Q_i is still consistent with the team.
(In 03b, the continuous-action version needed twin critics to stay stable; this discrete version trained without them.)

In [ ]:
qmix = seed_runs("2s3z", "qmix")
describe_smac_networks(qmix[0])

In [ ]:
show_video(smac_evolution(qmix[0], k=4), fps=6, width=720)

## 4. Side by side on `2s3z` (3 seeds each)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for name, runs in (("MAPPO", mappo), ("VDN", vdn), ("QMIX", qmix)):
    plot_seeds(runs, "train_win_rate", ax[0], name, COL[name.lower()])
    plot_seeds(runs, "return", ax[1], name, COL[name.lower()])
ax[0].set_title("battles won while training"); ax[1].set_title("return of the greedy policy")
for a in ax: a.legend()
plt.tight_layout()

*Win rates during training include exploration: VDN/QMIX take a random action 5% of the time, MAPPO samples from its policy.*

## 5. `MMM`: 10 agents and a healer

A bigger, mixed army. The Medivac cannot attack and healing never appears in the reward - it only helps indirectly, by keeping teammates alive.
Does it learn to heal, and does the algorithm matter?

In [ ]:
mmm = {a: seed_runs("MMM", a) for a in ("mappo", "vdn", "qmix")}
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, runs in mmm.items():
    plot_seeds(runs, "train_win_rate", ax[0], a.upper(), COL[a])
    plot_seeds(runs, "heal_frac", ax[1], a.upper(), COL[a])
    plot_seeds(runs, "healing", ax[2], a.upper(), COL[a])
ax[0].set_title("MMM: battles won while training"); ax[1].set_title("Medivac: share of its steps spent healing")
ax[2].set_title("Medivac: hp healed per battle")
for x in ax: x.legend(fontsize=8)
plt.tight_layout()

In [ ]:
show_video(smac_video(load_smac_algo(mmm["mappo"][0]["dir"] / "final.pt"), "MMM", title="MAPPO on MMM (final)")[0] +
           smac_video(load_smac_algo(mmm["qmix"][0]["dir"] / "final.pt"), "MMM", title="QMIX on MMM (final)")[0],
           fps=6, width=720)

## 6. Under the hood: who gets the credit?

All three learn from the same team reward, but they turn it into each agent's learning signal differently. Replay one `2s3z` battle and plot,
for every agent, the signal that would update it: MAPPO's **advantage**, and QMIX's per-agent **utility gap** Q_i(chosen) - mean Q_i.
Dots mark the steps where that agent dealt damage.

In [ ]:
algo = load_smac_algo(mappo[1]["dir"] / "final.pt")
sig = episode_signals(algo, "2s3z", seed=3)
plt.figure(figsize=(11, 3.6))
plot_episode_credit(sig, "2s3z", title="MAPPO: each agent's advantage during one battle")
print(f"similarity of the agents' signals: {signal_similarity(sig):.2f}   (1 = all agents get the same signal)")

MAPPO's signals move together: every agent is judged by how the *team* did. QMIX's per-agent values differ from agent to agent.

In [ ]:
algo = load_smac_algo(qmix[0]["dir"] / "final.pt")
sig = episode_signals(algo, "2s3z", seed=3)
plt.figure(figsize=(11, 3.6))
plot_episode_credit(sig, "2s3z", title="QMIX: each agent's own utility gap during one battle")
print(f"similarity of the agents' signals: {signal_similarity(sig):.2f}")

In [ ]:
import numpy as np
summary = load_credit_summary()
print("how similar are the agents' learning signals? (final policies, 8 battles each; 1 = identical)")
for algo in ("mappo", "vdn", "qmix"):
    for m in ("2s3z", "mmm"):
        vals = [float(r["similarity"]) for r in summary if r["algo"] == algo and f"_{m}_" in r["run"]]
        if vals:
            print(f"  {algo.upper():6s} {m:5s}: mean {np.mean(vals):.2f}   per seed {[round(v, 2) for v in vals]}")

## Takeaways
* **All three learn to win** on both maps from a single shared reward - including the Medivac, which learns to heal although healing is never rewarded.
* **MAPPO is a strong, simple default**: fastest on `MMM` and in 2 of 3 seeds on `2s3z` - but one seed collapsed late in training.
  **VDN was the most consistent** across seeds; **QMIX** learned more slowly and was less steady on `MMM`.
* **Credit differs under the hood**: MAPPO gives every agent the team's verdict (signal similarity 0.85 on `2s3z`, 1.00 on `MMM`), VDN/QMIX give
  per-agent credit (~0.2-0.4). On these maps that did not decide the outcome; on harder maps (sparse rewards, very uneven roles)
  value decomposition is the usual choice.
* **Harder domains cost compute**: SMAClite runs ~100-200 steps/s per process, so even these small maps needed parallel workers and ~1M steps.